<a href="https://colab.research.google.com/github/safakatakancelik/portfolio-public/blob/main/notebooks/building_attention_from_scratch/attention_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$



## Building Attention

1. Vectorization of an Example Input
2. Projecting Vectors into Q, K, V Spaces
3. Calculating Attention Scores
4. Applying Causal Masking
5. Calculating Attention Weights with Softmax
5. Calculating Scaled Dot Product Attention with Value Vectors
6. Summary of Step 1-6 for Scaled Dot Product Attention
7. Rewrite Attention in PyTorch and use a real embedding model


### Learning Goals (attention specific)
- understanding attention at fundamental levels
- bridge from numpy prototype to production-style PyTorch code
- understand causal masking for autoregressive generation
- understand attention scores/attention logits, attention weights and what they represent
- see why √d_k scaling prevents softmax saturation
- understand what Q, K, V matrixes represent and how that is achieved
- understand why training the Q, K, V projection matrices is crucial for meaningful attention
- understand the math behind these operations (intuition + core operations)

In [1]:
# Installing packages of interest
!pip install torch
!pip install torchtext
!pip install transformers
## these are needed for the PyTorch implementation

In [2]:
# Importing packages, libraries
import numpy as np # using numpy for linear algebra
from scipy.special import softmax # importing a softmax function instead of implementing from scratch

## Step 1: Vectorization of an Example Input

In [3]:
# random seed for best practices
np.random.seed(42)

# setting model dimensions
d_model = 8
d_k = 8

# array of english words that makes up our example sentence
english_words = ["the", "cat", "sat", "on", "the", "mat"]

# building the vocabulary with random vectors
unique_words = dict.fromkeys(english_words)
vocab = {w: np.random.randn(d_model) for w in unique_words}

# building the input vector of X based on our sequence
X = np.array([vocab[w] for w in english_words])  # (6, 8)


print("Tokens:", english_words)
print(np.round(X, 2))
## here we have 6x8 matrix for our input X.
## 6 vectors with 8 dimensions representing the 6 words in the sentence "the cat sat on the mat"
## we can try predicting a 7th word or one of the existing words with masking and/or train our matrices later.
## main goal here is building the attention and understanding how it works.

Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']
[[ 0.5  -0.14  0.65  1.52 -0.23 -0.23  1.58  0.77]
 [-0.47  0.54 -0.46 -0.47  0.24 -1.91 -1.72 -0.56]
 [-1.01  0.31 -0.91 -1.41  1.47 -0.23  0.07 -1.42]
 [-0.54  0.11 -1.15  0.38 -0.6  -0.29 -0.6   1.85]
 [ 0.5  -0.14  0.65  1.52 -0.23 -0.23  1.58  0.77]
 [-0.01 -1.06  0.82 -1.22  0.21 -1.96 -1.33  0.2 ]]


## Step 2: Projecting Vectors into Q, K, V spaces

In [4]:
# initialize projection matrices to be trained
W_q = np.random.randn(d_model, d_k) * 0.01
W_k = np.random.randn(d_model, d_k) * 0.01
W_v = np.random.randn(d_model, d_k) * 0.01


# project embeddings into Q, K, V spaces
Q = X @ W_q
K = X @ W_k
V = X @ W_v
## here we do matrix multiplications with projection matrices we initialized and the input X
## Q, K, V correspond to random 6,8 matrices at this point. They are all different from each other and the input X.
## During training the weights W_q, W_k, W_v will be trained to have meaningful representations for Q, K, V based on our data.

## Step 3: Calculating Attention Scores

$$attention \ scores = \frac{QK^T}{\sqrt{d_k}}$$




In [5]:
## Attention scores
attention_scores = Q @ K.T / np.sqrt(d_k)  # (6, 6)
## We follow the formula as it is,
## matrix multiplication between query and key matrices, key matrix transposed to allow the multiplication,
## scaled with the key vector dimension preventing softmax saturation later on


print("Tokens:", english_words)
print("\nScores shape:", attention_scores.shape)
print(np.round(attention_scores, 5))
## for each word combination
## now we have (6, 6) matrix which has attention scores, attention logits
## how much each word attends to another, raw similarity scores

Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']

Scores shape: (6, 6)
[[-5.80e-04  1.02e-03  2.60e-04  2.60e-04 -5.80e-04 -8.00e-05]
 [ 8.40e-04 -1.29e-03 -3.50e-04 -2.80e-04  8.40e-04 -6.10e-04]
 [-3.30e-04 -4.10e-04  5.90e-04 -3.60e-04 -3.30e-04 -2.70e-04]
 [ 2.10e-04 -2.60e-04 -3.60e-04  1.90e-04  2.10e-04 -1.30e-04]
 [-5.80e-04  1.02e-03  2.60e-04  2.60e-04 -5.80e-04 -8.00e-05]
 [ 3.40e-04 -6.80e-04  3.80e-04 -1.20e-03  3.40e-04 -1.40e-04]]


## Step 4: Apply Causal Masking

In [6]:
# prepare a matrix for masking
lower_triangle_array_mask = np.tril(np.ones_like(attention_scores))
# array([[1., 0., 0., 0., 0., 0.],
#        [1., 1., 0., 0., 0., 0.],
#        [1., 1., 1., 0., 0., 0.],
#        [1., 1., 1., 1., 0., 0.],
#        [1., 1., 1., 1., 1., 0.],
#        [1., 1., 1., 1., 1., 1.]])

# set upper triangle to negative infinity to allow causal masking
attention_scores[lower_triangle_array_mask == 0] = - np.inf

np.round(attention_scores, 5)
# now we have our lower triangle with existing values
# upper triangle with negative infinities
## this is to ease operations when calculating probabilities
## it allows us to force each word in a sequence to only attend to itself and the scores before it (causal masking)

array([[-0.00058,     -inf,     -inf,     -inf,     -inf,     -inf],
       [ 0.00084, -0.00129,     -inf,     -inf,     -inf,     -inf],
       [-0.00033, -0.00041,  0.00059,     -inf,     -inf,     -inf],
       [ 0.00021, -0.00026, -0.00036,  0.00019,     -inf,     -inf],
       [-0.00058,  0.00102,  0.00026,  0.00026, -0.00058,     -inf],
       [ 0.00034, -0.00068,  0.00038, -0.0012 ,  0.00034, -0.00014]])

## Step 5: Calculating Attention Weights with Softmax

$$\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)$$

In [7]:
# Calculating probabilities with softmax
attention_weights = softmax(attention_scores, axis=-1)


np.round(attention_weights, 5)
## we have turned our scores into normalized probabilities
## now we have a (6, 6) matrix of attention weights (probabilities).
## Each row sums to 1 and represents how much each word attends to itself and preceding words, enforced by the causal mask.

array([[1.     , 0.     , 0.     , 0.     , 0.     , 0.     ],
       [0.50053, 0.49947, 0.     , 0.     , 0.     , 0.     ],
       [0.33324, 0.33321, 0.33355, 0.     , 0.     , 0.     ],
       [0.25007, 0.24995, 0.24992, 0.25006, 0.     , 0.     ],
       [0.19987, 0.20019, 0.20004, 0.20004, 0.19987, 0.     ],
       [0.16675, 0.16658, 0.16676, 0.16649, 0.16675, 0.16667]])

## Step 6: Calculating Scaled Dot Product Attention with Value Vectors



$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [8]:
# Matrix multiplication of attention weights and the Value projection matrix
scaled_dot_product_attention = attention_weights @ V
# (6, 6) @ (6, 8) -> (6, 8)
# back to original shape: 6 words, 8 dimensions


np.round(scaled_dot_product_attention, 5)
## This (6, 8) matrix is the final output of the single-head attention mechanism.


## Weighted sum of the Value vectors
## Our attention weights(probabilities) are used to weight


## Each row is the aggregated contextual representation for each input word
## Due to causal masking, each row's representation only incorporates information from itself and preceding words.
## For decoding, for example, 5th row, index 4, is the numerical representation to decode the 6th word "mat"

array([[-1.729e-02, -2.582e-02, -9.390e-03, -1.452e-02,  1.186e-02,
        -2.819e-02,  5.610e-03, -3.521e-02],
       [-4.530e-03, -4.313e-02, -4.980e-03, -5.970e-03, -1.688e-02,
        -7.260e-03, -7.000e-05, -1.092e-02],
       [ 9.300e-04, -2.637e-02,  1.610e-03,  9.480e-03, -7.160e-03,
        -1.174e-02,  5.350e-03, -6.210e-03],
       [-6.000e-05, -1.983e-02,  1.300e-04,  2.010e-03, -8.910e-03,
         1.890e-03, -2.800e-04, -7.090e-03],
       [-3.500e-03, -2.103e-02, -1.770e-03, -1.290e-03, -4.770e-03,
        -4.120e-03,  9.000e-04, -1.270e-02],
       [-3.690e-03, -2.806e-02, -1.090e-03, -1.061e-02, -1.337e-02,
        -2.440e-03,  1.360e-03, -6.630e-03]])

##Step 7: Summary of Step 1-6 for Scaled Dot Product Attention

$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$



- Step 1: Vectorization (X) of an Example Input: The input sentence words are converted into numerical vector representations.

- Step 2: Projecting Vectors into Q, K, V Spaces: The input vectors (X) are transformed into Query (Q), Key (K), and Value (V) matrices by multiplying them with randomly initialized projection matrices.

- Step 3: Calculating Attention Scores: Raw similarity scores between each Query vector and all Key vectors are computed, scaled by the square root of the dimension of the key vectors.

- Step 4: Applying Causal Masking: A causal mask is applied to the attention scores to ensure that each word can only attend to itself and the words that come before it in the sequence.

- Step 5: Calculating Attention Weights with Softmax: The masked attention scores are passed through a softmax function to convert them into normalized probability distributions, representing attention weights.

- Step 6: Calculating Scaled Dot Product Attention with Value Vectors: The attention weights are multiplied by the Value matrix to produce the final contextualized output for each word, which is a weighted sum of the Value vectors.

## Step 8: Rewrite Attention in PyTorch and use a real embedding model


After building with NumPy, PyTorch version is straightforward, easy to read and don't need an extensive elaboration.

We are using existing functions and objects of PyTorch which handles everything in the background. They're highly optimized and we can easily use parameters to change settings instead of implementing everything from scratch.

Understanding earlier steps allows making sense of the actual operations and better decision-making.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Load BERT-Tiny for a real embedding model
tokenizer = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
tiny_model = AutoModel.from_pretrained("prajjwal1/bert-tiny")

# Extract strictly the Word Embedding layer
real_embedding_layer = tiny_model.embeddings.word_embeddings
D_MODEL = real_embedding_layer.embedding_dim

print(f"Real Vocab Size: {real_embedding_layer.num_embeddings} words")
print(f"Real Dimension (d_model): {D_MODEL} features per word")

# Tokenize our example sentence
sentence = "the cat sat on the mat"
tokens = tokenizer(sentence, return_tensors="pt")
input_ids = tokens["input_ids"]

print(f"\nReal Tokens: {tokenizer.convert_ids_to_tokens(input_ids[0])}")
print(f"Real Token IDs: {input_ids}")

# Pass the IDs through the real pre-trained embeddings
X = real_embedding_layer(input_ids)
print(f"\nShape of our Input X: {X.shape} -> (Batch, Seq_Len, d_model)")
print(X)

In [10]:
class PyTorchSelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # Project the real embeddings into Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # FlashAttention handles the scaling, causal masking, Softmax, and V multiplication!
        output = F.scaled_dot_product_attention(
            query=q,
            key=k,
            value=v,
            is_causal=True
        )

        return output

attention_layer = PyTorchSelfAttention(d_model=D_MODEL)

final_output = attention_layer(X)

print(f"Final Attention Output Shape: {final_output.shape}")
print(f"Row 5 (Used to predict the 6th word):\n{final_output[0, 4, :10]}... (showing first 10 dims)")

Final Attention Output Shape: torch.Size([1, 8, 128])
Row 5 (Used to predict the 6th word):
tensor([ 0.0155, -0.0149,  0.0156, -0.0085, -0.0117,  0.0006,  0.0242,  0.0021,
         0.0025,  0.0271], grad_fn=<SliceBackward0>)... (showing first 10 dims)


In [11]:
#### DECODE
# We need an LM Head (Unembedding Matrix)
# In a real model, this is trained. As our projection matrices parameters, this too is for demonstration purposes.
# It maps from our features (128) -> our entire dictionary (30,522)
lm_head = nn.Linear(D_MODEL, real_embedding_layer.num_embeddings)
## this is to bridge the gap between contextual embedding to vocabulary probability


# Let's say we want to predict the word that comes AFTER "the cat sat on the"
# In BERT-Tiny, index is 5 because of the [CLS] token at the start
vector_to_decode = final_output[0, 5, :]  # Shape: (128)

# Project the vector into Vocabulary Space
# This gives us 30,522 raw scores (logits)
logits = lm_head(vector_to_decode)  # Shape: (30522)

# Find the index of the highest score!
# torch.argmax looks at all 30,522 scores and returns the ID of the biggest one
predicted_token_id = torch.argmax(logits)

# Use the tokenizer to translate that ID back into English
predicted_word = tokenizer.decode(predicted_token_id)

print(f"Predicted Token ID: {predicted_token_id}")
print(f"The model predicts the next word is: '{predicted_word}'")


Predicted Token ID: 18989
The model predicts the next word is: 'launches'


---


Here we are at the end of a notebook that implements attention with NumPy and then rewrites in PyTorch, while explaining what each steps does and elaborates on certain points.


Making of this notebook was not a linear process, it required lots of questioning and thinking, reading, making wrong assumptions, aha moments and so on. Not all of these can be put into a single notebook. However, as a person I've learned a lot in making of this notebook.

If it helps anyone out there, that'd just be amazing. Read around, don't attach to a single source, try yourself, question and repeat.

Also make sure to balance things and learn in an optimized way.

You should never reinvent fire. Some things are good to try building from scratch for the learning processes. Because for example in my case I will build and test more complex systems that uses these, and it makes sense for me, maybe it doesn't for you, and you can use PyTorch or other implementations right away.

Make your own decisions.

Pay attention <3